# Integracion: camara + ArUco + clasificacion + StockFish

Este cuadernillo integra los primeros modulos del proyecto sin usar todavia el brazo robotico.

Flujo esperado:
1. Capturar el tablero desde DroidCam por IP.
2. Detectar 4 marcadores ArUco en las esquinas.
3. Enderezar la imagen del tablero.
4. Segmentar 64 casillas.
5. Clasificar cada casilla con el modelo de PyTorch.
6. Convertir la matriz detectada a un tablero de `python-chess`.
7. Pedir a StockFish la jugada de negras.
8. Actualizar el tablero interno y mostrar la jugada que luego hara el brazo.


## 1. Instalacion de librerias

Ejecuta esta celda solo si falta alguna libreria. Para ArUco se necesita `opencv-contrib-python`, no solo `opencv-python`.

In [ ]:
# Ejecuta solo si hace falta:
# !pip install opencv-contrib-python numpy matplotlib torch torchvision python-chess pillow


## 2. Imports y configuracion general

Ajusta `CAMERA_INDEX`, `MODEL_PATH`, `CLASS_NAMES` y `ARUCO_DICT` segun tu computadora, tu modelo y tus marcadores.

In [2]:
from pathlib import Path
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image

import chess
import chess.engine
import chess.svg
from IPython.display import display, SVG

ROOT = Path.cwd()

# DroidCam por USB normalmente aparece como una webcam local.
# Si no funciona, prueba 0, 1, 2, 3 con la celda de prueba de camaras.
CAMERA_INDEX = 0
CAMERA_WIDTH = 1280
CAMERA_HEIGHT = 720

# Debe coincidir con el diccionario usado para imprimir tus ArUco.
ARUCO_DICT = cv2.aruco.DICT_4X4_50 # aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
WARP_SIZE = 800
CELL_MARGIN = 0.04

# Ruta esperada del modelo exportado desde ModeloClasificacion.ipynb.
MODEL_PATH = ROOT / "Modelo de Clasificacion" / "modelo_ajedrez_real.pth"

# Si existe dataset/train, el notebook intentara leer las clases desde ahi.
# Si no existe, edita esta lista para que tenga el MISMO orden usado al entrenar.
CLASS_NAMES = [
    "bb", "bk", "bn", "bp", "bq", "br",
    "empty",
    "wb", "wk", "wn", "wp", "wq", "wr",
]

STOCKFISH_CANDIDATES = [
    ROOT / "StockFish" / "stockfish-windows-x86-64-avx2.exe",
    ROOT / "StockFish" / "stockfish" / "stockfish-windows-x86-64-avx2.exe",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)


Dispositivo: cuda


## 3. Prueba de camaras

Usa esto para encontrar el indice correcto de DroidCam. Se mostrara una ventana por cada indice disponible; pulsa cualquier tecla para avanzar.

In [6]:
def probar_indices_camara(max_index=5):
    disponibles = []
    for idx in range(max_index + 1):
        cap = cv2.VideoCapture(idx, cv2.CAP_DSHOW)
        ok, frame = cap.read()
        if ok:
            disponibles.append(idx)
            cv2.putText(frame, f"Indice {idx}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 0), 3)
            cv2.imshow("Prueba de camara", frame)
            cv2.waitKey(800)
        cap.release()
    cv2.destroyAllWindows()
    print("Indices disponibles:", disponibles)
    return disponibles

#probar_indices_camara(5)


## 4. Deteccion ArUco y enderezado del tablero

Esta parte esta basada en tu cuadernillo `Dataset/Deteccion_Aruco.ipynb`, pero convertida en funciones para usarla en tiempo real.

In [ ]:
def ordenar_puntos(pts):
    pts = np.array(pts, dtype="float32")
    suma = pts.sum(axis=1)
    resta = np.diff(pts, axis=1)

    ordenados = np.zeros((4, 2), dtype="float32")
    ordenados[0] = pts[np.argmin(suma)]      # top-left
    ordenados[2] = pts[np.argmax(suma)]      # bottom-right
    ordenados[1] = pts[np.argmin(resta)]     # top-right
    ordenados[3] = pts[np.argmax(resta)]     # bottom-left
    return ordenados


def detectar_aruco(imagen_bgr):
    gray = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2GRAY)
    aruco_dict = cv2.aruco.getPredefinedDictionary(ARUCO_DICT)
    params = cv2.aruco.DetectorParameters()

    if hasattr(cv2.aruco, "ArucoDetector"):
        detector = cv2.aruco.ArucoDetector(aruco_dict, params)
        corners, ids, rejected = detector.detectMarkers(gray)
    else:
        corners, ids, rejected = cv2.aruco.detectMarkers(gray, aruco_dict, parameters=params)

    return corners, ids


def obtener_esquinas_tablero_desde_aruco(corners, ids):
    if ids is None or len(ids) < 4:
        raise ValueError("Se necesitan 4 marcadores ArUco visibles para detectar el tablero.")

    datos = []
    for marker_corners, marker_id in zip(corners, ids.flatten()):
        pts = marker_corners.reshape(4, 2)
        datos.append({
            "id": int(marker_id),
            "corners": pts,
            "centro": pts.mean(axis=0),
        })

    datos = sorted(datos, key=lambda d: d["centro"][1])
    arriba = sorted(datos[:2], key=lambda d: d["centro"][0])
    abajo = sorted(datos[2:4], key=lambda d: d["centro"][0])

    marker_tl = arriba[0]
    marker_tr = arriba[1]
    marker_bl = abajo[0]
    marker_br = abajo[1]

    # Orden de esquinas de OpenCV para cada marcador:
    # 0 top-left, 1 top-right, 2 bottom-right, 3 bottom-left.
    board_tl = marker_tl["corners"][2]
    board_tr = marker_tr["corners"][3]
    board_br = marker_br["corners"][0]
    board_bl = marker_bl["corners"][1]

    return ordenar_puntos(np.array([board_tl, board_tr, board_br, board_bl], dtype="float32"))


def enderezar_tablero(imagen_bgr, board_corners, size=WARP_SIZE):
    origen = ordenar_puntos(board_corners)
    destino = np.array([
        [0, 0],
        [size - 1, 0],
        [size - 1, size - 1],
        [0, size - 1],
    ], dtype="float32")
    matriz = cv2.getPerspectiveTransform(origen, destino)
    return cv2.warpPerspective(imagen_bgr, matriz, (size, size))


def detectar_y_enderezar(imagen_bgr, dibujar=False):
    corners, ids = detectar_aruco(imagen_bgr)
    board_corners = obtener_esquinas_tablero_desde_aruco(corners, ids)
    tablero = enderezar_tablero(imagen_bgr, board_corners)

    debug = imagen_bgr.copy()
    if dibujar:
        cv2.aruco.drawDetectedMarkers(debug, corners, ids)
        cv2.polylines(debug, [board_corners.astype(int)], True, (0, 255, 0), 3)

    return tablero, board_corners, debug


def dibujar_grid(tablero_bgr, n=8):
    img = tablero_bgr.copy()
    h, w = img.shape[:2]
    for i in range(n + 1):
        x = int(i * w / n)
        y = int(i * h / n)
        cv2.line(img, (x, 0), (x, h), (0, 255, 0), 2)
        cv2.line(img, (0, y), (w, y), (0, 255, 0), 2)
    return img


## 5. Captura en tiempo real con DroidCam

Pulsa `ESPACIO` cuando los 4 ArUco esten detectados. Pulsa `q` para cancelar.

In [4]:
def abrir_camara(index=CAMERA_INDEX):
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)
    if not cap.isOpened():
        raise RuntimeError(f"No se pudo abrir la camara con indice {index}.")
    return cap


def capturar_tablero_desde_camara(index=CAMERA_INDEX):
    cap = abrir_camara(index)
    ultimo_tablero = None
    ultimo_frame = None

    print("Ventana activa: ESPACIO captura, q cancela.")
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                continue

            ultimo_frame = frame.copy()
            texto = "Buscando 4 ArUco"
            color = (0, 0, 255)
            debug = frame.copy()

            try:
                tablero, corners, debug = detectar_y_enderezar(frame, dibujar=True)
                ultimo_tablero = tablero
                texto = "ArUco OK - ESPACIO para capturar"
                color = (0, 255, 0)
            except Exception:
                ultimo_tablero = None

            cv2.putText(debug, texto, (25, 45), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3)
            cv2.imshow("DroidCam - captura tablero", debug)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                raise KeyboardInterrupt("Captura cancelada por usuario.")
            if key == 32 and ultimo_tablero is not None:
                return ultimo_frame, ultimo_tablero
    finally:
        cap.release()
        cv2.destroyAllWindows()


# frame_bgr, tablero_recto = capturar_tablero_desde_camara(CAMERA_INDEX)
# plt.figure(figsize=(6, 6))
# plt.imshow(cv2.cvtColor(dibujar_grid(tablero_recto), cv2.COLOR_BGR2RGB))
# plt.axis("off");


## 6. Segmentacion de las 64 casillas

La fila superior de la imagen enderezada se interpreta como la fila 8 del ajedrez: `a8` a `h8`.

In [ ]:
def nombre_casilla_ajedrez(fila, col):
    columnas = ["a", "b", "c", "d", "e", "f", "g", "h"]
    filas = ["8", "7", "6", "5", "4", "3", "2", "1"]
    return columnas[col] + filas[fila]


def segmentar_casillas(tablero_bgr, n=8, margin=CELL_MARGIN):
    h, w = tablero_bgr.shape[:2]
    cell_h = h // n
    cell_w = w // n
    casillas = []

    for fila in range(n):
        for col in range(n):
            y1 = fila * cell_h
            y2 = (fila + 1) * cell_h
            x1 = col * cell_w
            x2 = (col + 1) * cell_w

            mx = int(cell_w * margin)
            my = int(cell_h * margin)
            crop = tablero_bgr[y1 + my:y2 - my, x1 + mx:x2 - mx]

            casillas.append({
                "casilla": nombre_casilla_ajedrez(fila, col),
                "fila": fila,
                "columna": col,
                "imagen": crop,
            })

    return casillas


def mostrar_casillas(casillas):
    fig, axes = plt.subplots(8, 8, figsize=(10, 10))
    for item in casillas:
        fila = item["fila"]
        col = item["columna"]
        axes[fila, col].imshow(cv2.cvtColor(item["imagen"], cv2.COLOR_BGR2RGB))
        axes[fila, col].set_title(item["casilla"], fontsize=8)
        axes[fila, col].axis("off")
    plt.tight_layout()
    plt.show()


# casillas = segmentar_casillas(tablero_recto)
# mostrar_casillas(casillas)


## 7. Modelo de clasificacion

La arquitectura replica tu `ChessCNN` de `ModeloClasificacion.ipynb`. Si cambias la arquitectura en el notebook de entrenamiento, debes cambiarla aqui tambien.

In [ ]:
class ChessCNN(nn.Module):
    def __init__(self, num_classes=13):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.fc(self.conv(x))


preprocess = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])


def inferir_class_names():
    candidatos = [
        ROOT / "Modelo de Clasificacion" / "dataset" / "train",
        ROOT / "dataset" / "train",
    ]
    for carpeta in candidatos:
        if carpeta.exists():
            clases = sorted([p.name for p in carpeta.iterdir() if p.is_dir()])
            if clases:
                return clases
    return CLASS_NAMES


def cargar_modelo(model_path=MODEL_PATH):
    class_names = inferir_class_names()
    model = ChessCNN(num_classes=len(class_names)).to(device)

    if not Path(model_path).exists():
        raise FileNotFoundError(
            f"No existe el modelo: {model_path}\n"
            "Primero exporta el .pth desde ModeloClasificacion.ipynb o ajusta MODEL_PATH."
        )

    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()
    print("Modelo cargado:", model_path)
    print("Clases:", class_names)
    return model, class_names


def predecir_casilla(imagen_bgr, model, class_names):
    imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(imagen_rgb)
    x = preprocess(pil_img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]
        idx = int(torch.argmax(probs).item())

    return class_names[idx], float(probs[idx].item())


def clasificar_tablero(casillas, model, class_names):
    predicciones = {}
    for item in casillas:
        etiqueta, confianza = predecir_casilla(item["imagen"], model, class_names)
        predicciones[item["casilla"]] = {
            "label": etiqueta,
            "confidence": confianza,
            "fila": item["fila"],
            "columna": item["columna"],
        }
    return predicciones


# model, class_names = cargar_modelo()


## 8. Mapeo de etiquetas a piezas de ajedrez

Esta celda es la mas importante para conectar tu modelo con `python-chess`. Si tus carpetas tienen otros nombres, agrega esos nombres al diccionario.

In [ ]:
LABEL_TO_PIECE = {
    # Vacio
    "empty": None, "vacio": None, "blank": None, "none": None, "0": None,

    # Piezas negras
    "bp": "p", "black_pawn": "p", "peon_negro": "p", "peonnegro": "p",
    "bn": "n", "black_knight": "n", "caballo_negro": "n", "caballonegro": "n",
    "bb": "b", "black_bishop": "b", "alfil_negro": "b", "alfilnegro": "b",
    "br": "r", "black_rook": "r", "torre_negra": "r", "torrenegra": "r",
    "bq": "q", "black_queen": "q", "dama_negra": "q", "reina_negra": "q",
    "bk": "k", "black_king": "k", "rey_negro": "k", "reynegro": "k",

    # Piezas blancas
    "wp": "P", "white_pawn": "P", "peon_blanco": "P", "peonblanco": "P",
    "wn": "N", "white_knight": "N", "caballo_blanco": "N", "caballoblanco": "N",
    "wb": "B", "white_bishop": "B", "alfil_blanco": "B", "alfilblanco": "B",
    "wr": "R", "white_rook": "R", "torre_blanca": "R", "torreblanca": "R",
    "wq": "Q", "white_queen": "Q", "dama_blanca": "Q", "reina_blanca": "Q",
    "wk": "K", "white_king": "K", "rey_blanco": "K", "reyblanco": "K",
}


def normalizar_label(label):
    return str(label).strip().lower().replace(" ", "_").replace("-", "_")


def label_a_symbol(label):
    key = normalizar_label(label)
    if key not in LABEL_TO_PIECE:
        raise KeyError(
            f"No hay mapeo para la etiqueta '{label}'. "
            "Agregala en LABEL_TO_PIECE antes de continuar."
        )
    return LABEL_TO_PIECE[key]


def predicciones_a_board(predicciones, turn=chess.BLACK):
    board = chess.Board(None)
    matriz = []

    for fila in range(8):
        fila_matriz = []
        for col in range(8):
            square_name = nombre_casilla_ajedrez(fila, col)
            label = predicciones[square_name]["label"]
            symbol = label_a_symbol(label)
            fila_matriz.append(symbol if symbol is not None else ".")

            if symbol is not None:
                square = chess.parse_square(square_name)
                board.set_piece_at(square, chess.Piece.from_symbol(symbol))

        matriz.append(fila_matriz)

    board.turn = turn
    board.castling_rights = 0
    board.ep_square = None
    return board, matriz


def mostrar_matriz(matriz):
    print("Matriz detectada: filas 8 a 1, columnas a a h")
    print("   a b c d e f g h")
    for i, fila in enumerate(matriz):
        rank = 8 - i
        print(rank, " ", " ".join(fila))


def mostrar_board(board):
    display(SVG(chess.svg.board(board=board, size=420)))
    print("FEN:", board.fen())
    status = board.status()
    if status != chess.STATUS_VALID:
        print("Advertencia: python-chess marca esta posicion como no completamente valida. Status:", status)


## 9. StockFish

Esta celda abre StockFish usando `python-chess`. Por ahora siempre pedimos jugada para negras.

In [ ]:
def encontrar_stockfish():
    for path in STOCKFISH_CANDIDATES:
        if path.exists():
            return path
    raise FileNotFoundError("No se encontro StockFish. Revisa STOCKFISH_CANDIDATES.")


def abrir_stockfish():
    path = encontrar_stockfish()
    print("StockFish:", path)
    return chess.engine.SimpleEngine.popen_uci(str(path))


def pedir_jugada_negras(board, engine, time_limit=0.5):
    board_engine = board.copy(stack=False)
    board_engine.turn = chess.BLACK

    if board_engine.is_game_over():
        print("La partida ya termino segun python-chess.")
        return None, board_engine

    result = engine.play(board_engine, chess.engine.Limit(time=time_limit))
    move = result.move
    san = board_engine.san(move)
    board_engine.push(move)

    print("Jugada de negras:", move.uci(), "SAN:", san)
    print("Movimiento fisico pendiente para el brazo:", move.uci())
    return move, board_engine


# engine = abrir_stockfish()
# move, board_actualizado = pedir_jugada_negras(board, engine)
# mostrar_board(board_actualizado)
# engine.quit()


## 10. Pipeline completo de una captura

Esta funcion hace una iteracion completa: captura, ArUco, segmentacion, clasificacion, matriz, StockFish y actualizacion interna. Despues tu mueves fisicamente la jugada indicada y haces tu jugada blanca; luego ejecutas otra vez.

In [ ]:
def ejecutar_turno_negras(model, class_names, engine, camera_index=CAMERA_INDEX, time_limit=0.5):
    frame_bgr, tablero_recto = capturar_tablero_desde_camara(camera_index)

    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(dibujar_grid(tablero_recto), cv2.COLOR_BGR2RGB))
    plt.title("Tablero enderezado")
    plt.axis("off")
    plt.show()

    casillas = segmentar_casillas(tablero_recto)
    predicciones = clasificar_tablero(casillas, model, class_names)
    board, matriz = predicciones_a_board(predicciones, turn=chess.BLACK)

    mostrar_matriz(matriz)
    mostrar_board(board)

    fen_manual = input("Si necesitas corregir la posicion, pega FEN aqui. Enter para continuar: ").strip()
    if fen_manual:
        board = chess.Board(fen_manual)
        board.turn = chess.BLACK
        mostrar_board(board)

    move, board_actualizado = pedir_jugada_negras(board, engine, time_limit=time_limit)
    if board_actualizado is not None:
        mostrar_board(board_actualizado)

    return {
        "frame": frame_bgr,
        "tablero_recto": tablero_recto,
        "casillas": casillas,
        "predicciones": predicciones,
        "board_detectado": board,
        "move": move,
        "board_actualizado": board_actualizado,
    }


## 11. Ejecutar partida por turnos

Instrucciones:
1. Carga modelo y StockFish.
2. Ejecuta una iteracion.
3. Mueve fisicamente la jugada negra indicada.
4. Haz tu jugada blanca fisicamente.
5. Ejecuta otra iteracion.

In [ ]:
# 1) Cargar modelo y motor
# model, class_names = cargar_modelo()
# engine = abrir_stockfish()

# 2) Ejecutar una jugada de negras desde la posicion vista por camara
# resultado = ejecutar_turno_negras(model, class_names, engine, camera_index=CAMERA_INDEX, time_limit=0.5)

# 3) Cuando termines la sesion
# engine.quit()


## 12. Prueba sin camara usando una imagen ya enderezada

Util para depurar clasificacion y StockFish sin abrir DroidCam.

In [ ]:
def probar_con_imagen_enderezada(image_path=ROOT / "Dataset" / "tablero_enderezado.jpg"):
    tablero = cv2.imread(str(image_path))
    if tablero is None:
        raise FileNotFoundError(image_path)

    casillas = segmentar_casillas(tablero)
    predicciones = clasificar_tablero(casillas, model, class_names)
    board, matriz = predicciones_a_board(predicciones, turn=chess.BLACK)
    mostrar_matriz(matriz)
    mostrar_board(board)
    return board, predicciones


# board_prueba, predicciones_prueba = probar_con_imagen_enderezada()
# move, board_prueba_actualizado = pedir_jugada_negras(board_prueba, engine)
# mostrar_board(board_prueba_actualizado)
